
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 03 Lab - Modularize PySpark Code

#### Duration: ~15-20 Minutes

By the end of this lab, you will practice analyzing a PySpark script by breaking it down into smaller, reusable functions, and assessing how well their changes improve the code's clarity and ease of maintenance.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](https://files.training.databricks.com/binder/prod_main/devops-essentials-for-data-engineering-en_us-2.2.2/images/20261012T170439Z/DevOps Essentials for Data Engineering/Course Notebooks/Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

Run the following cell to configure your working environment for this course. 

##### The notebook "02 Demo - Modularizing PySpark Code - REQUIRED" sets up the catalogs for this course. If you have not run this notebook, the catalogs will not be available.

In [0]:
%run ../Includes/Classroom-Setup-Common

Run the cell below to view your current dev_ops catalog and schema. 

  Confirm the following:
- The catalog is your unique catalog name (shown above).
- The current schema is **dev_ops**.

In [0]:
%sql
SELECT current_catalog(), current_schema()

## B. Review the Provided PySpark Code

1. Run the cell below to preview the **samples.nyctaxi.trips** table. Confirm the table exists and view the data.

    Notice the following:
    - All columns are in lower case
    - **trip_distance** is currently in miles

In [0]:
%sql
SELECT * 
FROM samples.nyctaxi.trips 
LIMIT 10;

2. You have been provided with the following PySpark script that performs the following:

   a. Reads from the **samples.nyctaxi.trips** table.

   b. Creates a new column named **trip_distance_km** that converts **trip_distance** to kilometers and rounds it to two decimal places.

   c. Converts all of the column names to uppercase.

   d. Saves the DataFrame as a table named **nyc_lab_solution_table** in your specific catalog (`my_catalog`).

   Run the cell below and confirm that the **nyc_lab_solution_table** table was created with all uppercase column names and the new **trip_distance_km** column.

In [0]:
## Run the code to view the la catalog the table is being written to.
print(my_catalog)

In [0]:
# Import necessary libraries
from pyspark.sql import functions as F

# Load the data and create a new column named trip_distance_km
new_taxi = (spark
            .read
            .table("samples.nyctaxi.trips")
            .withColumn("trip_distance_km", F.round(F.col("trip_distance") * 1.60934, 2))
        )


## Upper case all columns
new_taxi = new_taxi.select([F.col(col).alias(col.upper()) for col in new_taxi.columns])


## Save the table to the your catalog
(new_taxi
 .write
 .mode('overwrite')
 .saveAsTable(f'{my_catalog}.dev_ops.nyc_lab_solution_table')
)

## View the final table
display(spark.table(f'{my_catalog}.dev_ops.nyc_lab_solution_table'))

## C. Modularize the PySpark Code

1. Your task is to take the provided Spark code from above and break it down into modular functions. Each function should perform a specific part of the task, making it easier to test, reuse, and maintain.

    There are a variety of ways to solve this problem. For consistency in this example, create the following functions:

    - `convert_miles_to_km`: Converts a column from miles to kilometers and rounds the result to two decimal places.

    - `uppercase_column_names`: Converts all column names in the DataFrame to uppercase.

    - `load_data`: Reads the table.

    - `save_to_catalog`: Saves the DataFrame as a new table in your catalog.

**NOTE:** The `load_data` and `save_to_catalog` functions have already been created for you. 

**TO DO:** Create the `convert_miles_to_km` and `uppercase_column_names` in the cell below.

**HINT:** The solution functions can be found in **[./src_lab/lab_functions/transforms.py]($./src_lab/lab_functions/transforms.py)**.

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

## load_data has already been created for you
def load_data(table_name):
    return spark.read.table(table_name)

## save_to_catalog has been created for you
def save_to_catalog(df, catalog_name, schema_name, table_name):
    (df
     .write
     .mode('overwrite')
     .saveAsTable(f'{catalog_name}.{schema_name}.{table_name}')
    )

In [0]:
## convert_miles_to_km
<FILL-IN>

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
## convert_miles_to_km
def convert_miles_to_km(df, new_column_name, miles_column):
    return df.withColumn(new_column_name, F.round(F.col(miles_column) * 1.60934, 2))
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.textContent.trim();

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

In [0]:
## uppercase_columns_names
<FILL-IN>

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
## uppercase_columns_names
def uppercase_columns_names(df):
    return df.select([F.col(col).alias(col.upper()) for col in df.columns])
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.textContent.trim();

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

2. Run your functions to obtain the same results as the original PySpark code. The `save_to_catalog` function will name your new table **my_lab_table**. 

**NOTE:** If you are receiving a schema mismatch error that is because you are trying to overwrite a table you created with a different schema. Delete the table and recreate the table.

In [0]:
## Load table
df = load_data("samples.nyctaxi.trips")

## Convert miles to km
## TODO - Add your function here

## Upcase column
## TODO - Add your function here

## Save DataFrame as a table in your catalog
save_to_catalog(df, catalog_name = my_catalog, schema_name="dev_ops", table_name = "my_lab_table")

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
## Load table
df = load_data("samples.nyctaxi.trips")

## Convert miles to km
df = convert_miles_to_km(df, new_column_name = "trip_distance_km", miles_column = "trip_distance")

## Upcase column
df = uppercase_columns_names(df)

## Save DataFrame as a table in your catalog
save_to_catalog(df, catalog_name = my_catalog, schema_name="dev_ops", table_name = "my_lab_table")
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.textContent.trim();

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

3. Run the following cell to test that the original table created in cell 11 (**nyc_lab_solution_table**) is the same as your new table created by the functions above (**my_lab_table**). The test uses the PySpark `assertDataFrameEqual` method.

    If there is an error, it means the original table is not the same as your new table, and you need to fix your functions.

**NOTE:** When you run `assertDataFrameEqual`, you might see the message `Tried to attach usage logger pyspark.databricks.pandas.usage_logger, but an exception was raised: JVM wasn't initialised` in the cell output. It is safe to ignore. `assertDataFrameEqual` imports the pandas API on Spark (`pyspark.pandas`) internally, which tries to attach a Databricks usage logger. The logger needs a direct JVM connection, which serverless compute does not provide. **It does not affect your code or test results**.

In [0]:
from pyspark.testing.utils import assertDataFrameEqual

# Read the tables (solution and your created table)
solution_df = spark.read.table(f"{my_catalog}.dev_ops.nyc_lab_solution_table")
user_df = spark.read.table(f"{my_catalog}.dev_ops.my_lab_table")

# Use assertDataFrameEqual to compare the two tables. Return an error if the tables are different.
assertDataFrameEqual(solution_df, user_df)

print("The tables are identical! Functions were created correctly.")


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
